**Mini Project 2**

In [1]:
# Installing necessary libraries
!pip install cupy
!pip install h5py

# Mounting Google Drive to access files
from google.colab import drive
drive.mount('/content/drive')

# Importing necessary libraries
import cupy as cp  # Importing Cupy for GPU-accelerated computations
import h5py  # Importing h5py for handling H5 file format

# Setting up file paths
file_path = "/content/drive/MyDrive/"
Mini_Project = file_path + 'Assignment 2 dataset.h5'  # Path to the H5 dataset file

# Loading data from the H5 file
with h5py.File(Mini_Project, 'r') as data:
    # Accessing dataset components
    trXX, trYY, tstXX, tstYY = data['trX'], data['trY'], data['tstX'], data['tstY']

    # Extracting data into arrays
    trX = trXX[:]  # Training data features
    trY = trYY[:]  # Training data labels
    tstX = tstXX[:]  # Test data features
    tstY = tstYY[:]  # Test data labels


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 11.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for cupy
  Running setup.py clean for cupy
Failed to build cupy
ERROR: Could not build wheels for cupy, which is required to install pyproject.toml-based projects
Mounted at /content/drive


In [16]:
# Hyperparameters
# These variables define various settings for the neural network training process.
learning_rates = [0.05, 0.1]
N_values = [50, 100]
batch_sizes = [10, 30]
epochs = 50

**Activation functions**

In [17]:
# Activation functions
# Functions used to introduce non-linearity into the network's layers.
def tanh(x):
    return cp.tanh(x)

def sigmoid(x):
    return 1 / (1 + cp.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def softmax(x):
    # Softmax function used in the output layer to normalize the output into probability distributions.
    exp_x = cp.exp(x - cp.max(x, axis=1, keepdims=True))
    return exp_x / cp.sum(exp_x, axis=1, keepdims=True)

**Loss functions and derivatives**


In [18]:
# Loss functions and derivatives
# These functions compute the discrepancy between predicted and actual values during training.
def cross_entropy_loss(y, y_hat):
    epsilon = 1e-15  # A small constant to avoid numerical instability in logarithm
    y_hat = cp.clip(y_hat, epsilon, 1 - epsilon)  # Clip values to avoid log(0) or log(1)
    return -cp.mean(y * cp.log(y_hat) + (1 - y) * cp.log(1 - y_hat))

def cross_entropy_derivative(y, y_hat):
    epsilon = 1e-15
    y_hat = cp.clip(y_hat, epsilon, 1 - epsilon)
    d_loss = -(y / y_hat - (1 - y) / (1 - y_hat))
    return d_loss / len(y)

**Initialize weights and biases**

In [19]:
# Function to initialize weights and biases for the neural network layers
def initialize_parameters(N):
    cp.random.seed(42)  # Set a seed for reproducibility
    WIH = cp.random.uniform(-0.1, 0.1, size=(N, 3 + 1))  # Weight matrix for input to hidden layer
    WHH = cp.random.uniform(-0.1, 0.1, size=(N, N))  # Weight matrix for hidden to hidden layer
    WHO = cp.random.uniform(-0.1, 0.1, size=(6, N + 1))  # Weight matrix for hidden to output layer
    return WIH, WHH, WHO

def one_hot_encode(labels):
    labels = labels.astype(int)
    one_hot = cp.eye(6)[labels]  # Create one-hot encoded vectors for categorical labels
    return one_hot

**Forward pass**

In [20]:
# Function to perform the forward pass through the neural network
def forward_pass(inputs, WIH, WHH, WHO):
    hidden_states = [cp.zeros((inputs.shape[0], WIH.shape[0]))]  # Initial hidden state

    for t in range(inputs.shape[1]):
        x_t = cp.hstack([inputs[:, t, :], cp.ones((inputs.shape[0], 1))])  # Concatenate input with bias
        h_t = tanh(cp.dot(x_t, WIH.T) + cp.dot(hidden_states[-1], WHH.T))  # Calculate hidden state
        hidden_states.append(h_t)  # Store hidden state for each time step

    output = sigmoid(cp.dot(cp.hstack([hidden_states[-1], cp.ones((inputs.shape[0], 1))]), WHO.T))  # Calculate output
    return hidden_states, output

**Backward Pass**

In [21]:
def backward_pass(inputs, labels, hidden_states, output, WIH, WHH, WHO, learning_rate):
    # Calculate error as the derivative of the cross-entropy loss
    error = cross_entropy_derivative(output, labels)
    error /= len(labels)  # Normalize error by the number of samples
    dWHO = cp.dot(error.T, cp.hstack([hidden_states[-1], cp.ones((inputs.shape[0], 1))]))  # Gradient for WHO

    # Initialize gradients for WHH and WIH
    dWHH = cp.zeros_like(WHH)
    dWIH = cp.zeros_like(WIH)

    # Calculate delta_output as the derivative of the sigmoid activation for the output layer
    delta_output = error * output * (1 - output)

    # Calculate delta_hidden as the derivative of the tanh activation for the hidden layer
    delta_hidden = cp.dot(delta_output, WHO[:, :-1]) * (1 - hidden_states[-1] ** 2)

    # Loop over time steps backward to compute gradients
    for t in range(inputs.shape[1], inputs.shape[1] - 10, -1):
        x_t = cp.hstack([inputs[:, t - 1, :], cp.ones((inputs.shape[0], 1))])

        # Compute gradients for WHH and WIH at each time step
        dWHH_t = cp.dot(delta_hidden.T, hidden_states[t - 1])  # Gradient for WHH at time step t
        dWIH_t = cp.dot(delta_hidden.T, x_t)  # Gradient for WIH at time step t

        # Accumulate gradients for each time step
        dWHH += dWHH_t
        dWIH += dWIH_t

        # Update delta_hidden for the previous time step using the recurrent weights WHH
        delta_hidden = cp.dot(delta_hidden, WHH) * (1 - hidden_states[t - 1] ** 2)

    # Normalize gradients by sequence length
    dWHH /= inputs.shape[1]
    dWIH /= inputs.shape[1]

    # Update weights and biases using the accumulated gradients and learning rate
    WHO -= learning_rate * dWHO
    WHH -= learning_rate * dWHH
    WIH -= learning_rate * dWIH

    return WIH, WHH, WHO  # Return updated weights and biases


**Training loop, Validation error calculation & Testing**

In [23]:
# Training loop
for learning_rate in learning_rates: # Iterate over different learning rates
    for N in N_values: # Iterate over different N values
        for batch_size in batch_sizes: # Iterate over different batch sizes
            WIH, WHH, WHO = initialize_parameters(N)  # Initialize weights and biases

# Training loop for multiple epochs
            for epoch in range(epochs): # Iterate over epochs
                total_loss = 0  # Initialize total loss for each epoch
                for i in range(0, trX.shape[0], batch_size):
                    batch_inputs = trX[i:i + batch_size]
                    batch_labels = cp.asarray(trY[i:i + batch_size])
                  # Perform the forward pass to obtain predictions
                    hidden_states, output = forward_pass(batch_inputs, WIH, WHH, WHO)

                    # Calculate cross-entropy loss with tanh and sigmoid activations
                    total_loss += cp.sum(-batch_labels * cp.log(output + 1e-10))

                    # Backward pass
                    WIH, WHH, WHO = backward_pass(batch_inputs, batch_labels, hidden_states, output, WIH, WHH, WHO, learning_rate)

                # Validation error (not part of the training loop)
                val_error = 0
                for j in range(0, 3000, 50):
                    val_inputs = trX[j:j + 50]
                    val_labels = cp.asarray(trY[j:j + 50])
                    _, val_output = forward_pass(val_inputs, WIH, WHH, WHO)
                    val_error += cp.sum(-val_labels * cp.log(val_output + 1e-10))

                average_loss = total_loss / (trX.shape[0] / batch_size)
                print(f"Epoch {epoch + 1}/{epochs}, Validation Error: {val_error / 3000}")

            # Testing
            _, test_output = forward_pass(tstX, WIH, WHH, WHO)
            test_predictions = cp.argmax(test_output, axis=1)
            # print(test_predictions,tstY)
            test_accuracy = cp.mean(test_predictions + 1 == cp.argmax(cp.asarray(tstY), axis=1))


            print(f"Learning Rate: {learning_rate}, N: {N}, Batch Size: {batch_size}")
            print("Test Accuracy:", test_accuracy)
            print()


Epoch 1/50, Validation Error: 3.8376418215734027
Epoch 2/50, Validation Error: 3.8376418215734027
Epoch 3/50, Validation Error: 3.8376418215734027
Epoch 4/50, Validation Error: 3.8376418215734027
Epoch 5/50, Validation Error: 3.8376418215734027
Epoch 6/50, Validation Error: 3.8376418215734027
Epoch 7/50, Validation Error: 3.8376418215734027
Epoch 8/50, Validation Error: 3.8376418215734027
Epoch 9/50, Validation Error: 3.8376418215734027
Epoch 10/50, Validation Error: 3.8376418215734027
Epoch 11/50, Validation Error: 3.8376418215734027
Epoch 12/50, Validation Error: 3.8376418215734027
Epoch 13/50, Validation Error: 3.8376418215734027
Epoch 14/50, Validation Error: 3.8376418215734027
Epoch 15/50, Validation Error: 3.8376418215734027
Epoch 16/50, Validation Error: 3.8376418215734027
Epoch 17/50, Validation Error: 3.8376418215734027
Epoch 18/50, Validation Error: 3.8376418215734027
Epoch 19/50, Validation Error: 3.8376418215734027
Epoch 20/50, Validation Error: 3.8376418215734027
Epoch 21/